In [1]:
import pandas as pd
import numpy as np
import os
from ukhls_variables import VARIABLE_MAP, DISABILITY_LABELS

# --- PATH CONFIGURATION ---
WAVES = ["o", "n", "m", "l", "k", "j", "a"]  # Scanning back to Wave L to find rotating variables
RAW_DIR = "../data/0_raw/ukhls"
PICKLE_DIR = "../data/2_pickle_ukhls_waves"

def run_standardized_ingestion():
    if not os.path.exists(PICKLE_DIR):
        os.makedirs(PICKLE_DIR)

    for w in WAVES:
        ind_path = os.path.join(RAW_DIR, f"{w}_indresp.tab")
        hh_path = os.path.join(RAW_DIR, f"{w}_hhresp.tab")
        
        if not os.path.exists(ind_path):
            print(f"Skipping Wave {w}: Individual response file not found.")
            continue

        print(f"--- Ingesting Wave {w} ---")
        
        # 1. SCAN HEADERS
        ind_headers = pd.read_csv(ind_path, sep='\t', nrows=0).columns.tolist()
        hh_exists = os.path.exists(hh_path)
        hh_headers = pd.read_csv(hh_path, sep='\t', nrows=0).columns.tolist() if hh_exists else []

        # 2. SELECT RELEVANT COLUMNS BASED ON VARIABLE_MAP
        ind_to_load = ['pidp']
        # Check for HIDP (Household Link) in Ind file
        hidp_col_name = next((c for c in [f"{w}_hidp", 'hidp'] if c in ind_headers), None)
        if hidp_col_name: ind_to_load.append(hidp_col_name)

        hh_to_load = []
        if hh_exists:
            # Check for HIDP in HH file
            hh_hidp = next((c for c in [f"{w}_hidp", 'hidp'] if c in hh_headers), None)
            if hh_hidp: hh_to_load.append(hh_hidp)

        # Distribute variables from VARIABLE_MAP to their respective files
        for base in VARIABLE_MAP.keys():
            if base == 'pidp': continue
            prefixed = f"{w}_{base}"
            if prefixed in ind_headers:
                ind_to_load.append(prefixed)
            elif prefixed in hh_headers:
                hh_to_load.append(prefixed)

        # 3. LOAD INDIVIDUAL AND HOUSEHOLD DATA
        df_ind = pd.read_csv(ind_path, sep='\t', usecols=ind_to_load, low_memory=False)
        
        if hh_to_load and len(hh_to_load) > 1:
            df_hh = pd.read_csv(hh_path, sep='\t', usecols=hh_to_load, low_memory=False)
            # Merge on Household ID
            df = pd.merge(df_ind, df_hh, on=hidp_col_name, how='left')
            print(f"   -> Successfully merged {len(hh_to_load)-1} household variables.")
        else:
            df = df_ind

        # 4. ONE-HOT ENCODE TARGET DISABILITIES
        # Disability categories are defined in DISABILITY_LABELS in ukhls_variables.py
        dis_prefix = f"{w}_disdif"
        dis_cols = [c for c in ind_headers if c.startswith(dis_prefix)]
        
        if dis_cols:
            # We load disability indicators separately to process them
            df_dis = pd.read_csv(ind_path, sep='\t', usecols=['pidp'] + dis_cols)
            
            encoded = 0
            for code, label in DISABILITY_LABELS.items():
                col = f"{dis_prefix}{code}"
                out_col = f"{w}_disability_{label}"
                # Skip duplicate labels (e.g. 'other' appears twice — take first)
                if out_col in df.columns:
                    continue
                if col in df_dis.columns:
                    clean_col = df_dis[col].mask(df_dis[col] < 0)
                    df[out_col] = np.where(clean_col.isna(), np.nan, (clean_col == 1).astype(float))
                else:
                    df[out_col] = np.nan
                encoded += 1
            
            print(f"   -> One-hot encoded {encoded} disability categories from DISABILITY_LABELS.")

        # 5. DATA CLEANING — Convert UKHLS missing codes (-9 to -1) to NaN
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].mask(df[numeric_cols] < 0)
        
        # 6. FEATURE ENGINEERING
        # These derive binary columns for variables where one_hot / binary encoding
        # is specified in ukhls_variables.py (wktrvfar -> drive_to_work, etc.)
        wktrv_col = f"{w}_wktrvfar"
        if wktrv_col in df.columns:
            df[f"{w}_drive_to_work"] = np.where(df[wktrv_col].isna(), np.nan, (df[wktrv_col] == 1).astype(float))
            print(f"   -> Engineered binary {w}_drive_to_work from {wktrv_col}")

        jbpl_col = f"{w}_jbpl"
        if jbpl_col in df.columns:
            df[f"{w}_work_at_home"] = np.where(df[jbpl_col].isna(), np.nan, (df[jbpl_col] == 1).astype(float))
            print(f"   -> Engineered binary {w}_work_at_home from {jbpl_col}")
            
        englang_col = f"{w}_englang"
        if englang_col in df.columns:
            # 2 = No. Any other valid response becomes True (1.0).
            df[f"{w}_englang_binary"] = np.where(df[englang_col].isna(), np.nan, (df[englang_col] != 2).astype(float))
            print(f"   -> Engineered binary {w}_englang_binary from {englang_col}")
        
        # 7. DTYPE OPTIMISATION
        for col in df.columns:
            if 'idp' in col:
                df[col] = df[col].fillna(0).astype(np.int64)
            elif 'sic' in col.lower() or 'soc' in col.lower() or df[col].dtype == 'object':
                df[col] = df[col].astype('category')
            else:
                df[col] = pd.to_numeric(df[col], downcast='float')

        # 8. SAVE OPTIMIZED PICKLE
        out_file = os.path.join(PICKLE_DIR, f"{w}_indresp_optimized.pkl")
        df.to_pickle(out_file, protocol=5)
        print(f"   -> Saved {len(df.columns)} variables for {len(df):,} rows to pickle.\n")

if __name__ == "__main__":
    run_standardized_ingestion()


--- Ingesting Wave o ---
   -> One-hot encoded 11 disability categories from DISABILITY_LABELS.
   -> Engineered binary o_drive_to_work from o_wktrvfar
   -> Engineered binary o_work_at_home from o_jbpl
   -> Saved 40 variables for 32,849 rows to pickle.

--- Ingesting Wave n ---
   -> One-hot encoded 11 disability categories from DISABILITY_LABELS.
   -> Engineered binary n_drive_to_work from n_wktrvfar
   -> Engineered binary n_work_at_home from n_jbpl
   -> Engineered binary n_englang_binary from n_englang
   -> Saved 39 variables for 35,471 rows to pickle.

--- Ingesting Wave m ---
   -> One-hot encoded 11 disability categories from DISABILITY_LABELS.
   -> Engineered binary m_drive_to_work from m_wktrvfar
   -> Engineered binary m_work_at_home from m_jbpl
   -> Saved 38 variables for 27,998 rows to pickle.

--- Ingesting Wave l ---
   -> One-hot encoded 11 disability categories from DISABILITY_LABELS.
   -> Engineered binary l_drive_to_work from l_wktrvfar
   -> Engineered binary 